# Bounded residual update and Mahalanobis screening on one split (seed 42)

This notebook fits the method on the seed-42 radiograph-grouped split of the DENTEX tooth crops for two frozen encoders, Dental ViT-L and BiomedCLIP, and produces the coverage and accuracy curves of Fig. 4.

For each encoder:
1. Reference: a logistic probe per condition on the frozen embeddings, scaled by one temperature `T` fitted on the validation split.
2. Residual update: a head Linear(d, 256), ReLU, Dropout(0.2), Linear(256, K) whose output is clipped to [-δ, δ] with δ = 1.5 and added to the reference logits, so that |final logit - reference logit| ≤ δ for every input and condition.
3. Screening: Mahalanobis distance of each embedding from the training embeddings, with a ridge of 1e-2 on the covariance.

Set `DATA_DIR` in Section 2, then run all cells.

## 1. Environment and device

In [ ]:
import sys, subprocess
def pipq(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p], check=False)
# Packages are listed in requirements.txt. Uncomment to install them into the active kernel.
# pipq("torch","torchvision","scikit-learn","pillow","matplotlib","tqdm",
#      "huggingface_hub","open_clip_torch","timm")

import torch
print("torch:", torch.__version__)
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


## 2. Data path

Set `DATA_DIR` to the DENTEX `quadrant-enumeration-disease` folder.

In [ ]:
import os, glob, json
DATA_DIR = "/path/to/DENTEX/training_data/quadrant-enumeration-disease"
assert os.path.isdir(DATA_DIR), f"Folder not found:\n{DATA_DIR}"

IMG_DIR = None
for root,_,files in os.walk(DATA_DIR):   # the xrays folder of the diagnosis subset
    if os.path.basename(root).lower()=="xrays" and any(f.lower().endswith((".png",".jpg")) for f in files):
        IMG_DIR = root; break
if IMG_DIR is None:                      # otherwise the first image folder that is not an output folder
    for root,_,files in os.walk(DATA_DIR):
        if any(s in root.lower() for s in ("_result","output","checkpoint")): continue
        if any(f.lower().endswith((".png",".jpg")) for f in files):
            IMG_DIR = root; break
assert IMG_DIR, "No image folder found."

ANN_PATH = None
for root,_,files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith(".json"):
            try: j=json.load(open(os.path.join(root,f)))
            except Exception: continue
            if isinstance(j,dict) and "annotations" in j and any("category_id_3" in a for a in j.get("annotations",[])):
                ANN_PATH=os.path.join(root,f); break
    if ANN_PATH: break
assert ANN_PATH, "Diagnosis JSON not found."
print("Images:", IMG_DIR, "\nLabels:", ANN_PATH)

OUTPUT_DIR = os.path.join(DATA_DIR, "_results_option1")
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 3. Tooth-level multi-label targets and radiograph-grouped split

In [ ]:
import numpy as np
coco = json.load(open(ANN_PATH))
cats3 = sorted(coco["categories_3"], key=lambda c:c["id"])
DX_NAMES=[c["name"] for c in cats3]; NUM_DX=len(DX_NAMES)
id2idx={c["id"]:i for i,c in enumerate(cats3)}
img_file={im["id"]:im["file_name"] for im in coco["images"]}

tooth={}
for a in coco["annotations"]:
    key=(a["image_id"],a.get("category_id_1",-1),a.get("category_id_2",-1))
    if key not in tooth:
        tooth[key]={"image_id":a["image_id"],"file":img_file[a["image_id"]],
                    "bbox":list(a["bbox"]),"y":np.zeros(NUM_DX,np.float32)}
    tooth[key]["y"][id2idx[a["category_id_3"]]]=1.0
    bx,by,bw,bh=tooth[key]["bbox"]; ax,ay,aw,ah=a["bbox"]
    x0,y0=min(bx,ax),min(by,ay); x1,y1=max(bx+bw,ax+aw),max(by+bh,ay+ah)
    tooth[key]["bbox"]=[x0,y0,x1-x0,y1-y0]
teeth=list(tooth.values()); Y=np.stack([t["y"] for t in teeth])
print(f"{len(teeth)} teeth | classes {DX_NAMES} | pos/class {Y.sum(0).astype(int).tolist()}")

SEED=42
def grouped_split(teeth,seed,vf=0.15,tf=0.15):
    ids=sorted({t["image_id"] for t in teeth}); rng=np.random.RandomState(seed); rng.shuffle(ids)
    n=len(ids); nte=int(n*tf); nva=int(n*vf)
    te=set(ids[:nte]); va=set(ids[nte:nte+nva]); tr_,va_,te_=[],[],[]
    for i,t in enumerate(teeth):
        (te_ if t["image_id"] in te else va_ if t["image_id"] in va else tr_).append(i)
    return tr_,va_,te_
train_idx,val_idx,test_idx=grouped_split(teeth,SEED)
def _im(ix): return {teeth[i]["image_id"] for i in ix}
assert _im(train_idx).isdisjoint(_im(val_idx)) and _im(train_idx).isdisjoint(_im(test_idx)) and _im(val_idx).isdisjoint(_im(test_idx))
print(f"split tr/va/te = {len(train_idx)}/{len(val_idx)}/{len(test_idx)} (no image shared)")


## 4. Tooth crops

Crops are returned as PIL images; each encoder applies its own preprocessing.

In [ ]:
from PIL import Image
PAD=0.15
_imc={}
def load_image(f):
    if f not in _imc: _imc[f]=Image.open(os.path.join(IMG_DIR, os.path.basename(f))).convert("RGB")
    return _imc[f]
def crop_tooth(img,bbox,pad=PAD):
    W,H=img.size; x,y,w,h=bbox; px,py=w*pad,h*pad
    x0=max(0,int(x-px));y0=max(0,int(y-py));x1=min(W,int(x+w+px));y1=min(H,int(y+h+py))
    return img if (x1<=x0 or y1<=y0) else img.crop((x0,y0,x1,y1))
def get_crop(i):
    t=teeth[i]; return crop_tooth(load_image(t["file"]), t["bbox"])
print("crop helper ready; example size:", get_crop(train_idx[0]).size)


## 5. BiomedCLIP encoder

BiomedCLIP is loaded through `open_clip`. This cell also defines the `embed(indices)` interface used for both encoders.

In [ ]:
import open_clip, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

def load_biomedclip():
    # open_clip can pull BiomedCLIP directly via the HF hub prefix
    model, _, preprocess = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    model = model.visual.to(DEVICE).eval()
    return model, preprocess

@torch.no_grad()
def embed_with(model, preprocess, indices, bs=32, l2norm=True):
    outs=[]
    batch=[]
    idxs=list(indices)
    for k in tqdm(range(0,len(idxs),bs), desc="embed", leave=False):
        ims=[preprocess(get_crop(i)) for i in idxs[k:k+bs]]
        x=torch.stack(ims).to(DEVICE)
        f=model(x)
        if isinstance(f,(tuple,list)): f=f[0]
        f=f.float().cpu()
        if l2norm: f=torch.nn.functional.normalize(f,dim=1)
        outs.append(f)
    return torch.cat(outs).numpy()

print("BiomedCLIP loader + embed interface ready.")


## 6. Dental ViT-L encoder

The checkpoint `Teeth_Visual_Experts_ViT_L_periapical_images_cls_7diseases.bin` is a jeonsworld ViT-pytorch model, not a Hugging Face Transformers model. The cell clones that repository for the model definition, builds the ViT-L/16 architecture, loads the weights and returns the 1,024-dimensional class-token embedding before the classification head. If the weights cannot be loaded, the cell prints the first checkpoint keys and the notebook continues with BiomedCLIP only.

In [ ]:
import os, torch
from huggingface_hub import hf_hub_download

DENTAL_OK = False
dental_model = None
dental_preprocess = None

try:
    # 1) clone the jeonsworld ViT implementation (GitHub, open)
    if not os.path.isdir("ViT-pytorch"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/jeonsworld/ViT-pytorch"], check=True)
    sys.path.insert(0, "ViT-pytorch")
    from models.modeling import VisionTransformer, CONFIGS

    # 2) download the dental ViT-L classification checkpoint
    ckpt = hf_hub_download(repo_id="Bryceee/Teeth_Visual_Experts_Models",
        filename="Teeth_Visual_Experts_ViT_L_periapical_images_cls_7diseases.bin")
    state = torch.load(ckpt, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state: state = state["state_dict"]
    print("checkpoint keys (first 12):")
    for k in list(state.keys())[:12]: print("   ", k, tuple(state[k].shape))

    # 3) build ViT-L/16 with 7 classes, try common training resolutions
    built=False
    for img_size in (224, 384):
        try:
            cfg = CONFIGS["ViT-L_16"]
            m = VisionTransformer(cfg, img_size=img_size, num_classes=7, zero_head=False)
            missing, unexpected = m.load_state_dict(state, strict=False)
            # success heuristic: most transformer weights loaded
            n_loaded = sum(1 for k in state if k in dict(m.named_parameters()))
            if n_loaded > 50:
                built=True; DENTAL_IMG=img_size; break
        except Exception as e:
            last_e=e
    assert built, "ViT-L state_dict did not align at 224 or 384"

    m = m.to(DEVICE).eval()

    # 4) preprocessing for jeonsworld ViT (ImageNet norm, square resize)
    import torchvision.transforms as T
    dental_preprocess = T.Compose([
        T.Resize((DENTAL_IMG, DENTAL_IMG)),
        T.ToTensor(),
        T.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
    ])

    # 5) wrap to expose the [CLS] embedding (pre-head). jeonsworld returns (logits, attn);
    #    the transformer trunk yields sequence -> take CLS token.
    class DentalEmbed(torch.nn.Module):
        def __init__(self, vit): super().__init__(); self.vit=vit
        @torch.no_grad()
        def forward(self, x):
            # transformer trunk: returns (encoded_seq, attn_weights)
            seq, _ = self.vit.transformer(x)
            return seq[:, 0]   # CLS token, 1024-d for ViT-L
    dental_model = DentalEmbed(m).to(DEVICE).eval()

    # sanity probe
    import torch as _t
    probe = dental_preprocess(get_crop(train_idx[0])).unsqueeze(0).to(DEVICE)
    with _t.no_grad(): d = dental_model(probe)
    print("dental ViT-L embedding dim:", tuple(d.shape))
    DENTAL_OK = True
    print("dental ViT-L loaded")
except Exception as e:
    print("dental ViT-L not loaded:", repr(e))
    print("Continuing with BiomedCLIP only.")


## 7. Frozen embeddings for each available encoder

In [ ]:
backbones = {}

# BiomedCLIP
bmc_model, bmc_pre = load_biomedclip()
backbones["biomedclip"] = lambda idx: embed_with(bmc_model, bmc_pre, idx)

# Dental ViT-L, if loaded
if DENTAL_OK:
    backbones["dental_vitl"] = lambda idx: embed_with(dental_model, dental_preprocess, idx, l2norm=True)

EMB = {}
for name, fn in backbones.items():
    print(f"\nExtracting with {name} ...")
    EMB[name] = {"tr": fn(train_idx), "va": fn(val_idx), "te": fn(test_idx)}
    print(f"  {name}: train {EMB[name]['tr'].shape}")
Ytr=Y[train_idx]; Yva=Y[val_idx]; Yte=Y[test_idx]


## 8. Reference: temperature-scaled logistic probe

A logistic-regression probe is fitted per condition on the training embeddings, and one temperature `T` is fitted on the validation split. The residual update is measured against this reference.

In [ ]:
from sklearn.linear_model import LogisticRegression
import torch, torch.nn as nn

def fit_baseline(Xtr, Ytr):
    probes=[]
    for c in range(NUM_DX):
        if len(np.unique(Ytr[:,c]))<2:
            probes.append(None)  # class absent in train
        else:
            probes.append(LogisticRegression(max_iter=1000, C=1.0).fit(Xtr, Ytr[:,c]))
    return probes
def baseline_logits(probes, X):
    out=np.zeros((len(X),NUM_DX),np.float32)
    for c,p in enumerate(probes):
        out[:,c]= p.decision_function(X) if p is not None else 0.0
    return out
def fit_temperature(logits, y):
    T=nn.Parameter(torch.ones(1)); z=torch.tensor(logits); yt=torch.tensor(y)
    opt=torch.optim.LBFGS([T], lr=0.1, max_iter=60); bce=nn.BCEWithLogitsLoss()
    def closure(): opt.zero_grad(); l=bce(z/T.clamp(min=0.05), yt); l.backward(); return l
    opt.step(closure); return float(T.detach().clamp(min=0.05))
def ece(y,p,nb=10):
    b=np.linspace(0,1,nb+1); e=0.0
    for lo,hi in zip(b[:-1],b[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()==0: continue
        e+=(m.sum()/len(p))*abs(p[m].mean()-y[m].mean())
    return e

baseline = {}
for name in EMB:
    pr = fit_baseline(EMB[name]["tr"], Ytr)
    zva = baseline_logits(pr, EMB[name]["va"])
    T   = fit_temperature(zva, Yva)
    baseline[name] = {"probes":pr, "T":T}
    print(f"{name}: temperature T={T:.3f}")


## 9. Clipped residual head and its training

`final_logit = reference_logit + clip(r(x), -δ, δ)`. Only the residual head is trained; the reference is kept fixed. The clip makes `|final_logit - reference_logit| ≤ δ` hold for every input.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DELTA = 1.5   # clip bound in logit space

class EnvelopeHead(nn.Module):
    def __init__(self, d, k, delta):
        super().__init__(); self.delta=delta
        self.net=nn.Sequential(nn.Linear(d,256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256,k))
    def residual(self,x): return torch.clamp(self.net(x), -self.delta, self.delta)
    def forward(self,x,base): return base + self.residual(x)

def train_envelope(name, epochs=60, lr=1e-3):
    Xtr=torch.tensor(EMB[name]["tr"]); Xva=torch.tensor(EMB[name]["va"])
    T=baseline[name]["T"]; pr=baseline[name]["probes"]
    btr=torch.tensor(baseline_logits(pr,EMB[name]["tr"])/T)
    bva=torch.tensor(baseline_logits(pr,EMB[name]["va"])/T)
    ytr=torch.tensor(Ytr); yva=torch.tensor(Yva)
    head=EnvelopeHead(Xtr.shape[1],NUM_DX,DELTA).to(DEVICE)
    opt=torch.optim.AdamW(head.parameters(),lr=lr,weight_decay=1e-4)
    crit=nn.BCEWithLogitsLoss()
    dl=DataLoader(TensorDataset(Xtr,btr,ytr),batch_size=64,shuffle=True)
    best=1e9; best_state=None
    for ep in range(epochs):
        head.train()
        for x,b,y in dl:
            x,b,y=x.to(DEVICE),b.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(); loss=crit(head(x,b),y); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            vl=crit(head(Xva.to(DEVICE),bva.to(DEVICE)), yva.to(DEVICE)).item()
        if vl<best: best=vl; best_state={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
    head.load_state_dict(best_state)
    return head

envelopes={n:train_envelope(n) for n in EMB}
print("trained envelopes for:", list(envelopes.keys()))


## 10. Reference and residual update: AUC, ECE and the bound check

In [ ]:
from sklearn.metrics import roc_auc_score
def macro_auc(yt,yp):
    v=[roc_auc_score(yt[:,c],yp[:,c]) for c in range(NUM_DX) if len(np.unique(yt[:,c]))==2]
    return float(np.mean(v)) if v else float("nan")

results={}
for name in EMB:
    pr=baseline[name]["probes"]; T=baseline[name]["T"]
    bte=baseline_logits(pr,EMB[name]["te"])/T
    p_base=1/(1+np.exp(-bte))
    head=envelopes[name].eval()
    with torch.no_grad():
        final=head(torch.tensor(EMB[name]["te"]).to(DEVICE),
                   torch.tensor(bte).to(DEVICE)).cpu().numpy()
    p_env=1/(1+np.exp(-final))
    # guarantee check
    max_dev=np.max(np.abs(final-bte))
    results[name]={
        "baseline_macroAUC":round(macro_auc(Yte,p_base),4),
        "envelope_macroAUC":round(macro_auc(Yte,p_env),4),
        "baseline_ECE":round(float(np.mean([ece(Yte[:,c],p_base[:,c]) for c in range(NUM_DX)])),4),
        "envelope_ECE":round(float(np.mean([ece(Yte[:,c],p_env[:,c]) for c in range(NUM_DX)])),4),
        "max_logit_deviation":round(float(max_dev),4),
        "delta":DELTA,
        "guarantee_holds":bool(max_dev<=DELTA+1e-4),
    }
    print(f"\n=== {name} ===")
    for k,v in results[name].items(): print(f"  {k}: {v}")


## 11. Coverage and accuracy under Mahalanobis screening (Fig. 4)

Test teeth are ranked by Mahalanobis distance from the training embeddings. Element-wise accuracy is computed on the retained fraction as the most distant teeth are removed.

In [ ]:
import matplotlib.pyplot as plt

def maha_scorer(Xtr):
    mu=Xtr.mean(0); cov=np.cov(Xtr.T)+np.eye(Xtr.shape[1])*1e-2
    ci=np.linalg.inv(cov)
    return lambda X: np.einsum("ni,ij,nj->n", X-mu, ci, X-mu)

fig, axes = plt.subplots(1, len(EMB), figsize=(5*len(EMB),4), squeeze=False)
for j,name in enumerate(EMB):
    score=maha_scorer(EMB[name]["tr"])(EMB[name]["te"])
    pr=baseline[name]["probes"]; T=baseline[name]["T"]
    bte=baseline_logits(pr,EMB[name]["te"])/T
    head=envelopes[name].eval()
    with torch.no_grad():
        final=head(torch.tensor(EMB[name]["te"]).to(DEVICE),
                   torch.tensor(bte).to(DEVICE)).cpu().numpy()
    p=1/(1+np.exp(-final))
    order=np.argsort(score)  # most in-distribution first
    fracs=np.linspace(0.3,1.0,15); accs=[]
    for fr in fracs:
        keep=order[:max(1,int(fr*len(order)))]
        accs.append(((p[keep]>0.5)==Yte[keep]).mean())
    ax=axes[0][j]
    ax.plot(fracs,accs,"o-",color="#1a3a5c")
    ax.set_title(f"{name}\nrefusal/accuracy"); ax.set_xlabel("kept fraction"); ax.set_ylabel("accuracy on kept")
    ax.grid(alpha=0.3)
    results[name]["refusal_curve"]={round(float(f),2):round(float(a),4) for f,a in zip(fracs,accs)}
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,"ood_refusal_curves.png"),dpi=130); plt.show()


## 12. Save results

In [ ]:
import json, datetime
summary={
    "timestamp":datetime.datetime.now().isoformat(timespec="seconds"),
    "device":DEVICE,"seed":SEED,"delta":DELTA,
    "backbones":list(EMB.keys()),
    "class_names":DX_NAMES,
    "split":{"train":len(train_idx),"val":len(val_idx),"test":len(test_idx)},
    "results":results,
    "notes":{
        "method":"temperature-scaled logistic probe, clipped residual head and Mahalanobis gate on frozen embeddings",
        "backbone_A":"dental ViT-L (periapical-pretrained, applied to panoramic tooth crops)",
        "backbone_B":"BiomedCLIP",
        "guarantee":"|final-baseline|<=delta by construction",
        "caveats":"705 radiographs; Periapical Lesion is the rarest condition",
    },
}
p=os.path.join(OUTPUT_DIR,f"option1_summary_seed{SEED}.json")
json.dump(summary,open(p,"w"),indent=2)
print("Saved ->",p)
print(json.dumps({n:{ "base_AUC":results[n]["baseline_macroAUC"],
                      "env_AUC":results[n]["envelope_macroAUC"],
                      "base_ECE":results[n]["baseline_ECE"],
                      "env_ECE":results[n]["envelope_ECE"],
                      "guarantee":results[n]["guarantee_holds"]} for n in results}, indent=2))


---
# Additional analyses on the seed-42 split

These cells use the reference probes, temperature and residual heads fitted above, on the 529 test teeth of the seed-42 split. Each reruns nothing upstream; run them after Section 12.

## 13. Operating points at threshold 0.5 (Table 4)

In [ ]:
# Threshold-0.5 operating points on the test split: reference probe versus bounded update
import numpy as np
def clinical(yt, p, thr=0.5):
    yp=(p>=thr).astype(int)
    tp=int(((yp==1)&(yt==1)).sum()); tn=int(((yp==0)&(yt==0)).sum())
    fp=int(((yp==1)&(yt==0)).sum()); fn=int(((yp==0)&(yt==1)).sum())
    sens=tp/(tp+fn) if tp+fn else np.nan
    spec=tn/(tn+fp) if tn+fp else np.nan
    ppv =tp/(tp+fp) if tp+fp else np.nan
    npv =tn/(tn+fn) if tn+fn else np.nan
    f1  =2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else np.nan
    return sens,spec,ppv,npv,f1

for name in EMB:
    pr=baseline[name]["probes"]; T=baseline[name]["T"]
    bte=baseline_logits(pr,EMB[name]["te"])/T
    p_base=1/(1+np.exp(-bte))
    head=envelopes[name].eval()
    with torch.no_grad():
        final=head(torch.tensor(EMB[name]["te"]).to(DEVICE),
                   torch.tensor(bte).to(DEVICE)).cpu().numpy()
    p_env=1/(1+np.exp(-final))
    print(f"\n=== {name} (test n={len(Yte)}) ===")
    print(f"{'condition':18s} {'':4s} sens  spec  ppv   npv   f1")
    for c,cn in enumerate(DX_NAMES):
        if len(np.unique(Yte[:,c]))<2: 
            print(f"{cn:18s}  (only one class in test, skipped)"); continue
        sb=clinical(Yte[:,c],p_base[:,c]); se=clinical(Yte[:,c],p_env[:,c])
        print(f"{cn:18s} base {sb[0]:.2f} {sb[1]:.2f} {sb[2]:.2f} {sb[3]:.2f} {sb[4]:.2f}")
        print(f"{'':18s} env  {se[0]:.2f} {se[1]:.2f} {se[2]:.2f} {se[3]:.2f} {se[4]:.2f}")
    mb=np.nanmean([clinical(Yte[:,c],p_base[:,c])[4] for c in range(NUM_DX) if len(np.unique(Yte[:,c]))==2])
    me=np.nanmean([clinical(Yte[:,c],p_env[:,c])[4]  for c in range(NUM_DX) if len(np.unique(Yte[:,c]))==2])
    print(f"  macro-F1: base {mb:.3f} -> env {me:.3f}")

## 14. Paired bootstrap of the macro-AUC difference

In [ ]:
# Paired tooth-level bootstrap (2,000 resamples) of the macro-AUC difference on the test split
import numpy as np
from sklearn.metrics import roc_auc_score
def macro_auc_idx(yt,yp,idx):
    v=[]
    for c in range(NUM_DX):
        y=yt[idx,c]
        if len(np.unique(y))==2: v.append(roc_auc_score(y,yp[idx,c]))
    return np.mean(v) if v else np.nan

for name in EMB:
    pr=baseline[name]["probes"]; T=baseline[name]["T"]
    bte=baseline_logits(pr,EMB[name]["te"])/T
    p_base=1/(1+np.exp(-bte))
    head=envelopes[name].eval()
    with torch.no_grad():
        final=head(torch.tensor(EMB[name]["te"]).to(DEVICE),
                   torch.tensor(bte).to(DEVICE)).cpu().numpy()
    p_env=1/(1+np.exp(-final))
    rng=np.random.default_rng(0); N=len(Yte); diffs=[]
    for _ in range(2000):
        idx=rng.integers(0,N,N)
        diffs.append(macro_auc_idx(Yte,p_env,idx)-macro_auc_idx(Yte,p_base,idx))
    diffs=np.array(diffs)
    lo,hi=np.quantile(diffs,.025),np.quantile(diffs,.975)
    print(f"{name:12s} macro-AUC gain {diffs.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  "
          f"-> {'significant' if (lo>0 or hi<0) else 'NOT significant (CI spans 0)'}")

## 15. Calibration comparison: temperature, Platt scaling and isotonic regression

In [ ]:
# Calibration comparison on the Dental ViT-L probe logits: none, temperature, isotonic and Platt
import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
name="dental_vitl"
pr=baseline[name]["probes"]; T=baseline[name]["T"]
zva=baseline_logits(pr,EMB[name]["va"]); zte=baseline_logits(pr,EMB[name]["te"])

def ece(y,p,nb=10):
    b=np.linspace(0,1,nb+1); e=0.0
    for lo,hi in zip(b[:-1],b[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum(): e+=(m.sum()/len(p))*abs(p[m].mean()-y[m].mean())
    return e

print(f"{'condition':18s} {'uncal':>6s} {'temp':>6s} {'isot':>6s} {'platt':>6s}")
rows=[]
for c,cn in enumerate(DX_NAMES):
    yv,yt=Yva[:,c],Yte[:,c]
    if len(np.unique(yv))<2: print(f"{cn:18s}  (one class in val, skipped)"); continue
    p_uncal=1/(1+np.exp(-zte[:,c]))
    p_temp =1/(1+np.exp(-zte[:,c]/T))
    iso=IsotonicRegression(out_of_bounds='clip').fit(1/(1+np.exp(-zva[:,c])),yv)
    p_iso=iso.predict(1/(1+np.exp(-zte[:,c])))
    pl=LogisticRegression().fit(zva[:,c].reshape(-1,1),yv); p_platt=pl.predict_proba(zte[:,c].reshape(-1,1))[:,1]
    r=(ece(yt,p_uncal),ece(yt,p_temp),ece(yt,p_iso),ece(yt,p_platt)); rows.append(r)
    print(f"{cn:18s} {r[0]:6.3f} {r[1]:6.3f} {r[2]:6.3f} {r[3]:6.3f}")
rows=np.array(rows)
print(f"{'MACRO':18s} {rows[:,0].mean():6.3f} {rows[:,1].mean():6.3f} {rows[:,2].mean():6.3f} {rows[:,3].mean():6.3f}")

## 16. Logit-based detectors versus the Mahalanobis score on Tufts (Table 7)

Maximum sigmoid probability, negative log-sum-exp of the reference logits (energy), and a temperature-only ODIN variant (T = 1000, no input perturbation) are computed from the multi-label reference probes and compared with the Mahalanobis score. The Tufts tooth-crop embeddings are read from the cache written by `04_tufts_shift_screening.ipynb`, so run that notebook first.

In [ ]:
# Detector comparison, DENTEX test teeth versus Tufts tooth crops: logit-based scores and Mahalanobis
import numpy as np, os
CACHE_DIR = os.path.join(OUTPUT_DIR, "emb_cache")   # Tufts embeddings written by 04_tufts_shift_screening.ipynb

def auroc(d_in,d_ood):
    s=np.concatenate([d_ood,d_in]); y=np.concatenate([np.ones(len(d_ood)),np.zeros(len(d_in))])
    o=np.argsort(s); r=np.empty_like(o,float); r[o]=np.arange(1,len(s)+1)
    npos,nneg=int(y.sum()),int((y==0).sum())
    return (r[y==1].sum()-npos*(npos+1)/2)/(npos*nneg)
def fpr95(d_in,d_ood):
    return float((d_in>=np.quantile(d_ood,0.05)).mean())
def maha_fit(Xtr):
    mu=Xtr.mean(0); cov=np.cov(Xtr.T)+np.eye(Xtr.shape[1])*1e-2; ci=np.linalg.inv(cov)
    return lambda X: np.einsum("ni,ij,nj->n",X-mu,ci,X-mu)

print(f"{'backbone':12s} {'detector':10s} {'AUROC':>7s} {'FPR95':>7s}")
for name in ["dental_vitl","biomedclip"]:
    Xtr=EMB[name]["tr"]; Xte=EMB[name]["te"]
    Xood=np.load(os.path.join(CACHE_DIR,f"tufts_toothcrop_emb_{name}.npy")).astype(np.float32)  # Tufts tooth crops
    # logits via the existing trained baseline probes
    pr=baseline[name]["probes"]
    def logits(X):
        out=np.zeros((len(X),len(pr)))
        for c,p in enumerate(pr): out[:,c]= p.decision_function(X) if p is not None else 0.0
        return out
    Lte,Lood=logits(Xte),logits(Xood)
    # MSP (OOD score = -max sigmoid prob)
    msp_in=-(1/(1+np.exp(-Lte))).max(1); msp_ood=-(1/(1+np.exp(-Lood))).max(1)
    # Energy
    en_in=-np.log(np.exp(Lte).sum(1)); en_ood=-np.log(np.exp(Lood).sum(1))
    # ODIN (temperature-only, T=1000)
    od_in=-(1/(1+np.exp(-Lte/1000))).max(1); od_ood=-(1/(1+np.exp(-Lood/1000))).max(1)
    # Mahalanobis distance on the embeddings
    sc=maha_fit(Xtr); mh_in,mh_ood=sc(Xte),sc(Xood)
    for det,(di,do) in [("MSP",(msp_in,msp_ood)),("Energy",(en_in,en_ood)),
                        ("ODIN",(od_in,od_ood)),("Mahalanobis",(mh_in,mh_ood))]:
        print(f"{name:12s} {det:10s} {auroc(di,do):7.3f} {fpr95(di,do):7.3f}")
    print()